In [1]:
!pip -q install transformers accelerate sentencepiece pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
import json
import random
import re
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Set seeds so results are more reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
DATA_PATH = "impostor_pair_dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (9, 12)


,q1_category,q2_category,q1_answers_json,q2_answers_json,shared_answers_json,q1_only_answers_json,q2_only_answers_json,n_q1_answers,n_q2_answers,n_shared,n_q1_only,n_q2_only
0,green vegetable,vegetable,"[""broccoli"", ""peas"", ""lettuce"", ""cucumber"", ""s...","[""carrot"", ""broccoli"", ""cabbage"", ""parsnips"", ...","[""beans"", ""broccoli"", ""brussels sprouts"", ""cab...","[""celery"", ""green pepper"", ""peppers"", ""spring ...","[""asparagus"", ""aubergine"", ""beetroot"", ""butter...",15,29,11,4,18
1,bird,water bird,"[""pigeon"", ""duck"", ""robin"", ""eagle"", ""seagull""...","[""duck"", ""swan"", ""penguin"", ""goose"", ""pelican""...","[""duck"", ""flamingo"", ""goose"", ""penguin"", ""seag...","[""blackbird"", ""bluebird"", ""canary"", ""chicken"",...","[""heron"", ""kingfisher"", ""pelican"", ""puffin"", ""...",27,11,6,21,5
2,part of the body,part of the face,"[""legs"", ""arm"", ""head"", ""hands"", ""ears"", ""mout...","[""nose"", ""eyes"", ""eyebrows"", ""mouth"", ""ears"", ...","[""ears"", ""eyes"", ""mouth"", ""nose"", ""skin"", ""tee...","[""ankle"", ""arm"", ""back"", ""bones"", ""brain"", ""ca...","[""beard"", ""cheek"", ""cheekbones"", ""cheeks"", ""ch...",41,22,7,34,15
3,farm animal,four-legged animal,"[""cow"", ""sheep"", ""pig"", ""chicken"", ""horse"", ""g...","[""dog"", ""cat"", ""lion"", ""horse"", ""tiger"", ""elep...","[""cow"", ""dog"", ""donkey"", ""goat"", ""horse"", ""pig...","[""bull"", ""chicken"", ""duck"", ""geese"", ""lamb"", ""...","[""antelope"", ""bear"", ""bison"", ""cat"", ""cheetah""...",15,35,7,8,28
4,building,religious building,"[""house"", ""flats"", ""offices"", ""church"", ""schoo...","[""church"", ""mosque"", ""temple"", ""synagogue"", ""c...","[""cathedral"", ""church"", ""mosque"", ""synagogue""]","[""apartment"", ""bungalow"", ""bus station"", ""cast...","[""chapel"", ""monastery"", ""temple""]",30,7,4,26,3


In [5]:
JSON_COLUMNS = [
    "q1_answers_json",
    "q2_answers_json",
    "shared_answers_json",
    "q1_only_answers_json",
    "q2_only_answers_json",
]

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return json.loads(x)

for col in JSON_COLUMNS:
    df[col] = df[col].apply(parse_json_list)

print("Finished parsing JSON columns.")
display(df.head(2))

Finished parsing JSON columns.


,q1_category,q2_category,q1_answers_json,q2_answers_json,shared_answers_json,q1_only_answers_json,q2_only_answers_json,n_q1_answers,n_q2_answers,n_shared,n_q1_only,n_q2_only
0,green vegetable,vegetable,"[broccoli, peas, lettuce, cucumber, spinach, g...","[carrot, broccoli, cabbage, parsnips, lettuce,...","[beans, broccoli, brussels sprouts, cabbage, c...","[celery, green pepper, peppers, spring onions]","[asparagus, aubergine, beetroot, butternut squ...",15,29,11,4,18
1,bird,water bird,"[pigeon, duck, robin, eagle, seagull, parrot, ...","[duck, swan, penguin, goose, pelican, flamingo...","[duck, flamingo, goose, penguin, seagull, swan]","[blackbird, bluebird, canary, chicken, crow, d...","[heron, kingfisher, pelican, puffin, stork]",27,11,6,21,5


In [6]:
row = df.iloc[0]

print("Q1 category:", row["q1_category"])
print("Q2 category:", row["q2_category"])
print("Q1 answers sample:", row["q1_answers_json"][:5])
print("Q2 answers sample:", row["q2_answers_json"][:5])
print("Shared answers sample:", row["shared_answers_json"][:5])

Q1 category: green vegetable
Q2 category: vegetable
Q1 answers sample: ['broccoli', 'peas', 'lettuce', 'cucumber', 'spinach']
Q2 answers sample: ['carrot', 'broccoli', 'cabbage', 'parsnips', 'lettuce']
Shared answers sample: ['beans', 'broccoli', 'brussels sprouts', 'cabbage', 'courgette']


In [7]:
def normalize_text(text):
    """
    Lowercase and remove extra whitespace/punctuation
    so answer comparisons are more stable.
    """
    text = str(text).strip().lower()
    text = re.sub(r"[\"'`]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def normalize_list(items):
    return [normalize_text(x) for x in items]

# Create normalized versions of all answer lists
for col in JSON_COLUMNS:
    norm_col = col.replace("_json", "_norm")
    df[norm_col] = df[col].apply(normalize_list)

display(df.head(2))

,q1_category,q2_category,q1_answers_json,q2_answers_json,shared_answers_json,q1_only_answers_json,q2_only_answers_json,n_q1_answers,n_q2_answers,n_shared,n_q1_only,n_q2_only,q1_answers_norm,q2_answers_norm,shared_answers_norm,q1_only_answers_norm,q2_only_answers_norm
0,green vegetable,vegetable,"[broccoli, peas, lettuce, cucumber, spinach, g...","[carrot, broccoli, cabbage, parsnips, lettuce,...","[beans, broccoli, brussels sprouts, cabbage, c...","[celery, green pepper, peppers, spring onions]","[asparagus, aubergine, beetroot, butternut squ...",15,29,11,4,18,"[broccoli, peas, lettuce, cucumber, spinach, g...","[carrot, broccoli, cabbage, parsnips, lettuce,...","[beans, broccoli, brussels sprouts, cabbage, c...","[celery, green pepper, peppers, spring onions]","[asparagus, aubergine, beetroot, butternut squ..."
1,bird,water bird,"[pigeon, duck, robin, eagle, seagull, parrot, ...","[duck, swan, penguin, goose, pelican, flamingo...","[duck, flamingo, goose, penguin, seagull, swan]","[blackbird, bluebird, canary, chicken, crow, d...","[heron, kingfisher, pelican, puffin, stork]",27,11,6,21,5,"[pigeon, duck, robin, eagle, seagull, parrot, ...","[duck, swan, penguin, goose, pelican, flamingo...","[duck, flamingo, goose, penguin, seagull, swan]","[blackbird, bluebird, canary, chicken, crow, d...","[heron, kingfisher, pelican, puffin, stork]"


In [23]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Loaded model:", MODEL_NAME)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded model: microsoft/Phi-3-mini-4k-instruct


In [24]:
def generate_chat_response(messages, max_new_tokens=32, temperature=0.7, do_sample=True):
    """
    Generate a response from the chat model.
    """
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=0.95 if do_sample else None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text

In [25]:
def build_few_shot_examples(train_rows, max_examples=4):
    """
    Build a few-shot prompt from other category pairs.
    Each example teaches the model:
    given q1 + q2, produce one q2-valid answer that could plausibly blend in.
    """
    if len(train_rows) == 0:
        return ""

    sampled = train_rows.sample(min(max_examples, len(train_rows)), random_state=SEED)

    examples = []
    for _, row in sampled.iterrows():
        # Prefer shared answers when available, since those are ideal impostor answers
        if len(row["shared_answers_json"]) > 0:
            target_answer = random.choice(row["shared_answers_json"])
        else:
            target_answer = random.choice(row["q2_answers_json"])

        ex = f"""Example
Visible category (q1): {row['q1_category']}
Impostor category (q2): {row['q2_category']}
Good impostor answer: {target_answer}"""
        examples.append(ex)

    return "\n\n".join(examples)

In [30]:
def clean_generated_answer(text):
    """
    Keep only the first short line and remove common formatting junk.
    """
    text = str(text).strip().split("\n")[0].strip()
    text = text.replace("Answer:", "").replace("Impostor answer:", "").strip()
    text = text.strip('"').strip("'").strip()

    # Remove leading bullets/numbers if the model adds them
    text = re.sub(r"^[\-\*\d\.\)\s]+", "", text).strip()

    return text

def generate_impostor_answer(q1, q2, few_shot_text):
    """
    Generate one impostor answer that fits q2 but could plausibly blend with q1.
    The answer should be a specific item, not a category label or explanation.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are the impostor in a one-word category game. "
                "You know the visible category q1 and your secret category q2. "
                "Your job is to give exactly ONE short answer that is a specific example from q2, "
                "but is also as plausible as possible for q1. "
                "Only give a concrete item someone could list in the category game. "
                "Do NOT give a general class, category label, part name, or explanation. "
                "Do NOT say words like 'leaf', 'stem', 'animal', 'fruit', or 'building' unless that exact word is a normal category answer. "
                "Good answers are specific items like 'spinach', 'lettuce', 'duck', 'church', or 'nose'. "
                "Bad answers are abstract labels like 'leaf', 'stem', 'body part', or 'vegetable'. "
                "Return only one noun or short noun phrase and nothing else."
            )
        },
        {
            "role": "user",
            "content": f"""{few_shot_text}

Now answer this round.

Visible category (q1): {q1}
Impostor category (q2): {q2}

Return exactly one short concrete answer that fits q2 and could also plausibly fit q1."""
        }
    ]

    raw_output = generate_chat_response(
        messages,
        max_new_tokens=8,
        temperature=0.6,
        do_sample=True
    )

    cleaned = clean_generated_answer(raw_output)
    cleaned = postprocess_impostor_answer(cleaned)

    return cleaned, raw_output

In [31]:
# Words that are too abstract / generic for this game
GENERIC_BAD_ANSWERS = {
    "thing", "item", "object", "stuff", "food", "animal", "bird", "vegetable",
    "fruit", "plant", "body part", "part", "face part", "building",
    "religious building", "farm animal", "four-legged animal", "green vegetable",
    "leaf", "stem", "root", "body", "face"
}

def postprocess_impostor_answer(answer):
    """
    Convert obviously weak or invalid generations into a cleaner form if possible.
    If the answer is too generic, return it as-is for evaluation,
    but this helper gives us one place to tighten rules later.
    """
    ans = normalize_text(answer)

    # If the model outputs a long phrase, keep only the first comma-separated part
    if "," in ans:
        ans = ans.split(",")[0].strip()

    # If it outputs an article, strip it
    ans = re.sub(r"^(a|an|the)\s+", "", ans).strip()

    return ans

In [32]:
def extract_judge_choice(text):
    """
    Extract the first valid choice from 1-4.
    """
    match = re.search(r"\b([1-4])\b", str(text))
    if match:
        return int(match.group(1))
    return None

def judge_impostor(q1, answers):
    """
    Judge which of the 4 answers least fits the visible category q1.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are judging a one-round impostor game. "
                "Exactly three answers fit the visible category well. "
                "Exactly one answer fits the visible category worse than the others and is from the impostor. "
                "Compare all four answers carefully and pick the single worst fit for the visible category. "
                "Reply with exactly one number: 1, 2, 3, or 4. Do not explain."
            )
        },
        {
            "role": "user",
            "content": f"""Visible category: {q1}

Answers:
1. {answers[0]}
2. {answers[1]}
3. {answers[2]}
4. {answers[3]}

Which answer is the single worst fit for the visible category?
Reply with only 1, 2, 3, or 4."""
        }
    ]

    raw_output = generate_chat_response(
        messages,
        max_new_tokens=4,
        temperature=0.0,
        do_sample=False
    )

    choice = extract_judge_choice(raw_output)
    return choice, raw_output

In [33]:
def classify_impostor_answer(answer, row):
    """
    Check whether the generated answer belongs to:
    - shared answers
    - q2-only answers
    - q2-valid-other
    - invalid
    Also reject generic abstract answers.
    """
    ans = normalize_text(answer)

    if ans in GENERIC_BAD_ANSWERS:
        return "invalid"

    shared = set(row["shared_answers_norm"])
    q2_only = set(row["q2_only_answers_norm"])
    q2_all = set(row["q2_answers_norm"])

    if ans in shared:
        return "shared"
    elif ans in q2_only:
        return "q2_only"
    elif ans in q2_all:
        return "q2_valid_other"
    else:
        return "invalid"

In [34]:
print("Number of category pairs:", len(df))
display(df[["q1_category", "q2_category"]])

Number of category pairs: 9


,q1_category,q2_category
0,green vegetable,vegetable
1,bird,water bird
2,part of the body,part of the face
3,farm animal,four-legged animal
4,building,religious building
5,building,human dwelling
6,herb,spice
7,animal,farm animal
8,insect,stinging insect


In [35]:
def simulate_one_round(row, few_shot_text):
    """
    Simulate one round from a single category pair.
    """
    q1 = row["q1_category"]
    q2 = row["q2_category"]

    # Sample 3 normal-player answers from q1
    normal_answers = random.sample(row["q1_answers_json"], 3)

    # Generate one impostor answer
    impostor_answer, impostor_raw = generate_impostor_answer(q1, q2, few_shot_text)

    # Check whether the impostor answer is valid according to q2 data
    impostor_type = classify_impostor_answer(impostor_answer, row)

    # Build the 4-answer round
    all_answers = normal_answers + [impostor_answer]
    labels = ["normal", "normal", "normal", "impostor"]

    paired = list(zip(all_answers, labels))
    random.shuffle(paired)

    shuffled_answers = [x[0] for x in paired]
    shuffled_labels = [x[1] for x in paired]

    true_impostor_position = shuffled_labels.index("impostor") + 1

    # Judge picks the odd one out
    judge_choice, judge_raw = judge_impostor(q1, shuffled_answers)

    judge_valid = judge_choice is not None
    judge_correct = (judge_choice == true_impostor_position) if judge_valid else None
    impostor_success = (not judge_correct) if judge_valid else None

    return {
        "q1": q1,
        "q2": q2,
        "normal_answers": normal_answers,
        "generated_impostor_answer": impostor_answer,
        "generated_impostor_raw": impostor_raw,
        "impostor_answer_type": impostor_type,
        "shown_answer_1": shuffled_answers[0],
        "shown_answer_2": shuffled_answers[1],
        "shown_answer_3": shuffled_answers[2],
        "shown_answer_4": shuffled_answers[3],
        "true_impostor_position": true_impostor_position,
        "judge_choice": judge_choice,
        "judge_raw_output": judge_raw,
        "judge_valid_output": judge_valid,
        "judge_correct": judge_correct,
        "impostor_success": impostor_success,
    }

In [37]:
for i in range(min(5, len(df))):
    test_row = df.iloc[i]

    # Use all OTHER rows as few-shot examples
    train_rows = df.drop(index=i).reset_index(drop=True)
    few_shot_text = build_few_shot_examples(train_rows, max_examples=4)

    result = simulate_one_round(test_row, few_shot_text)

    print("=" * 80)
    print("Q1:", result["q1"])
    print("Q2:", result["q2"])
    print("Normal answers:", result["normal_answers"])
    print("Generated impostor answer:", result["generated_impostor_answer"])
    print("Raw impostor output:", repr(result["generated_impostor_raw"]))
    print("Impostor answer type:", result["impostor_answer_type"])
    print("Shown answers:", [
        result["shown_answer_1"],
        result["shown_answer_2"],
        result["shown_answer_3"],
        result["shown_answer_4"],
    ])
    print("True impostor position:", result["true_impostor_position"])
    print("Judge choice:", result["judge_choice"])
    print("Judge raw output:", repr(result["judge_raw_output"]))
    print("Judge correct:", result["judge_correct"])

Q1: green vegetable
Q2: vegetable
Normal answers: ['brussels sprouts', 'broccoli', 'beans']
Generated impostor answer: spinach
Raw impostor output: 'spinach'
Impostor answer type: shared
Shown answers: ['beans', 'brussels sprouts', 'broccoli', 'spinach']
True impostor position: 4
Judge choice: 1
Judge raw output: '1'
Judge correct: False
Q1: bird
Q2: water bird
Normal answers: ['sparrow', 'hummingbird', 'vulture']
Generated impostor answer: egret
Raw impostor output: 'egret'
Impostor answer type: invalid
Shown answers: ['vulture', 'sparrow', 'egret', 'hummingbird']
True impostor position: 3
Judge choice: 4
Judge raw output: '4'
Judge correct: False
Q1: part of the body
Q2: part of the face
Normal answers: ['back', 'stomach', 'finger']
Generated impostor answer: cheek
Raw impostor output: 'cheek'
Impostor answer type: q2_only
Shown answers: ['finger', 'back', 'cheek', 'stomach']
True impostor position: 3
Judge choice: 4
Judge raw output: '4'
Judge correct: False
Q1: farm animal
Q2: four

In [38]:
ROUNDS_PER_PAIR = 10

all_results = []

for test_idx in tqdm(range(len(df)), total=len(df)):
    test_row = df.iloc[test_idx]

    # Leave-one-pair-out few-shot setup:
    # test on one pair, use the rest as examples
    train_rows = df.drop(index=test_idx).reset_index(drop=True)
    few_shot_text = build_few_shot_examples(train_rows, max_examples=4)

    for round_num in range(ROUNDS_PER_PAIR):
        round_result = simulate_one_round(test_row, few_shot_text)
        round_result["pair_index"] = test_idx
        round_result["round_num"] = round_num
        all_results.append(round_result)

results_df = pd.DataFrame(all_results)

print("Total simulated rounds:", len(results_df))
display(results_df.head())

  0%|          | 0/9 [00:00<?, ?it/s]

Total simulated rounds: 90


,q1,q2,normal_answers,generated_impostor_answer,generated_impostor_raw,impostor_answer_type,shown_answer_1,shown_answer_2,shown_answer_3,shown_answer_4,true_impostor_position,judge_choice,judge_raw_output,judge_valid_output,judge_correct,impostor_success,pair_index,round_num
0,green vegetable,vegetable,"[spinach, peppers, peas]",broccoli,broccoli,shared,peppers,peas,spinach,broccoli,4,1,1,True,False,True,0,0
1,green vegetable,vegetable,"[green pepper, cucumber, celery]",kale,kale,shared,cucumber,kale,celery,green pepper,2,4,4,True,False,True,0,1
2,green vegetable,vegetable,"[beans, broccoli, cucumber]",spinach,spinach,shared,spinach,cucumber,beans,broccoli,1,2,2,True,False,True,0,2
3,green vegetable,vegetable,"[cucumber, lettuce, courgette]",spinach,spinach,shared,lettuce,spinach,courgette,cucumber,2,4,4,True,False,True,0,3
4,green vegetable,vegetable,"[courgette, brussels sprouts, spinach]",spinach,spinach,shared,brussels sprouts,spinach,courgette,spinach,2,3,3,True,False,True,0,4


In [39]:
valid_judge_df = results_df[results_df["judge_valid_output"] == True].copy()

valid_answer_types = {"shared", "q2_only", "q2_valid_other"}
valid_judge_df["answer_is_valid_for_q2"] = valid_judge_df["impostor_answer_type"].isin(valid_answer_types)
valid_judge_df["valid_deception"] = valid_judge_df["answer_is_valid_for_q2"] & valid_judge_df["impostor_success"]

judge_accuracy = valid_judge_df["judge_correct"].mean()
impostor_success_rate = valid_judge_df["impostor_success"].mean()
shared_rate = (valid_judge_df["impostor_answer_type"] == "shared").mean()
q2_only_rate = (valid_judge_df["impostor_answer_type"] == "q2_only").mean()
invalid_rate = (valid_judge_df["impostor_answer_type"] == "invalid").mean()
valid_deception_rate = valid_judge_df["valid_deception"].mean()

print(f"Judge accuracy: {judge_accuracy:.3f}")
print(f"Impostor success rate: {impostor_success_rate:.3f}")
print(f"Shared answer rate: {shared_rate:.3f}")
print(f"Q2-only answer rate: {q2_only_rate:.3f}")
print(f"Invalid answer rate: {invalid_rate:.3f}")
print(f"Valid deception rate: {valid_deception_rate:.3f}")

Judge accuracy: 0.233
Impostor success rate: 0.767
Shared answer rate: 0.778
Q2-only answer rate: 0.067
Invalid answer rate: 0.156
Valid deception rate: 0.611


In [40]:
pair_summary = (
    valid_judge_df
    .groupby(["q1", "q2"])
    .agg(
        rounds=("round_num", "count"),
        judge_accuracy=("judge_correct", "mean"),
        impostor_success=("impostor_success", "mean"),
        shared_rate=("impostor_answer_type", lambda x: (x == "shared").mean()),
        q2_only_rate=("impostor_answer_type", lambda x: (x == "q2_only").mean()),
        invalid_rate=("impostor_answer_type", lambda x: (x == "invalid").mean()),
        valid_deception=("valid_deception", "mean"),
    )
    .reset_index()
)

display(pair_summary)

,q1,q2,rounds,judge_accuracy,impostor_success,shared_rate,q2_only_rate,invalid_rate,valid_deception
0,animal,farm animal,10,0.6,0.4,1.0,0.0,0.0,0.4
1,bird,water bird,10,0.4,0.6,0.6,0.2,0.2,0.4
2,building,human dwelling,10,0.2,0.8,1.0,0.0,0.0,0.8
3,building,religious building,10,0.1,0.9,1.0,0.0,0.0,0.9
4,farm animal,four-legged animal,10,0.0,1.0,1.0,0.0,0.0,1.0
5,green vegetable,vegetable,10,0.1,0.9,1.0,0.0,0.0,0.9
6,herb,spice,10,0.1,0.9,0.2,0.0,0.8,0.1
7,insect,stinging insect,10,0.4,0.6,1.0,0.0,0.0,0.6
8,part of the body,part of the face,10,0.2,0.8,0.2,0.4,0.4,0.4


In [41]:
print("=== Some INVALID impostor answers ===")
display(
    valid_judge_df[valid_judge_df["impostor_answer_type"] == "invalid"][
        ["q1", "q2", "generated_impostor_answer", "judge_correct", "impostor_success"]
    ].head(10)
)

print("\n=== Some SHARED impostor answers ===")
display(
    valid_judge_df[valid_judge_df["impostor_answer_type"] == "shared"][
        ["q1", "q2", "generated_impostor_answer", "judge_correct", "impostor_success"]
    ].head(10)
)

=== Some INVALID impostor answers ===


,q1,q2,generated_impostor_answer,judge_correct,impostor_success
14,bird,water bird,egret,False,True
19,bird,water bird,egret,False,True
20,part of the body,part of the face,ear,False,True
23,part of the body,part of the face,ear,False,True
26,part of the body,part of the face,ear,False,True
27,part of the body,part of the face,ear,False,True
60,herb,spice,rosemary,False,True
61,herb,spice,basil,False,True
62,herb,spice,dill,False,True
63,herb,spice,basil,False,True



=== Some SHARED impostor answers ===


,q1,q2,generated_impostor_answer,judge_correct,impostor_success
0,green vegetable,vegetable,broccoli,False,True
1,green vegetable,vegetable,kale,False,True
2,green vegetable,vegetable,spinach,False,True
3,green vegetable,vegetable,spinach,False,True
4,green vegetable,vegetable,spinach,False,True
5,green vegetable,vegetable,kale,False,True
6,green vegetable,vegetable,broccoli,False,True
7,green vegetable,vegetable,cucumber,True,False
8,green vegetable,vegetable,broccoli,False,True
9,green vegetable,vegetable,cucumber,False,True


In [46]:
results_df.to_csv("impostor_round_results.csv", index=False)
pair_summary.to_csv("impostor_pair_summary.csv", index=False)

print("Saved:")
print("- impostor_round_results.csv")
print("- impostor_pair_summary.csv")

Saved:
- impostor_round_results.csv
- impostor_pair_summary.csv


In [50]:
from google.colab import files

files.download("impostor_round_results.csv")
files.download("impostor_pair_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
results_df = pd.read_csv("impostor_round_results.csv")

print("Total rows:", len(results_df))

valid_df = results_df[results_df["judge_valid_output"] == True].copy()

valid_types = ["shared", "q2_only", "q2_valid_other"]

valid_df["answer_is_valid_for_q2"] = valid_df["impostor_answer_type"].isin(valid_types)
valid_df["valid_deception"] = valid_df["answer_is_valid_for_q2"] & valid_df["impostor_success"]

print("Judge accuracy:", valid_df["judge_correct"].mean())
print("Impostor success:", valid_df["impostor_success"].mean())
print("Shared rate:", (valid_df["impostor_answer_type"] == "shared").mean())
print("Q2-only rate:", (valid_df["impostor_answer_type"] == "q2_only").mean())
print("Invalid rate:", (valid_df["impostor_answer_type"] == "invalid").mean())
print("Valid deception rate:", valid_df["valid_deception"].mean())

display(valid_df.head(10))

Total rows: 90
Judge accuracy: 0.23333333333333334
Impostor success: 0.7666666666666667
Shared rate: 0.7777777777777778
Q2-only rate: 0.06666666666666667
Invalid rate: 0.15555555555555556
Valid deception rate: 0.6111111111111112


,q1,q2,normal_answers,generated_impostor_answer,generated_impostor_raw,impostor_answer_type,shown_answer_1,shown_answer_2,shown_answer_3,shown_answer_4,true_impostor_position,judge_choice,judge_raw_output,judge_valid_output,judge_correct,impostor_success,pair_index,round_num,answer_is_valid_for_q2,valid_deception
0,green vegetable,vegetable,"['spinach', 'peppers', 'peas']",broccoli,broccoli,shared,peppers,peas,spinach,broccoli,4,1,1,True,False,True,0,0,True,True
1,green vegetable,vegetable,"['green pepper', 'cucumber', 'celery']",kale,kale,shared,cucumber,kale,celery,green pepper,2,4,4,True,False,True,0,1,True,True
2,green vegetable,vegetable,"['beans', 'broccoli', 'cucumber']",spinach,spinach,shared,spinach,cucumber,beans,broccoli,1,2,2,True,False,True,0,2,True,True
3,green vegetable,vegetable,"['cucumber', 'lettuce', 'courgette']",spinach,spinach,shared,lettuce,spinach,courgette,cucumber,2,4,4,True,False,True,0,3,True,True
4,green vegetable,vegetable,"['courgette', 'brussels sprouts', 'spinach']",spinach,spinach,shared,brussels sprouts,spinach,courgette,spinach,2,3,3,True,False,True,0,4,True,True
5,green vegetable,vegetable,"['beans', 'peppers', 'lettuce']",kale,kale,shared,peppers,kale,beans,lettuce,2,1,1,True,False,True,0,5,True,True
6,green vegetable,vegetable,"['spring onions', 'spinach', 'green beans']",broccoli,broccoli,shared,green beans,spring onions,spinach,broccoli,4,2,2,True,False,True,0,6,True,True
7,green vegetable,vegetable,"['peas', 'green beans', 'brussels sprouts']",cucumber,cucumber,shared,brussels sprouts,cucumber,peas,green beans,2,2,2,True,True,False,0,7,True,False
8,green vegetable,vegetable,"['peppers', 'celery', 'green beans']",broccoli,broccoli,shared,celery,broccoli,green beans,peppers,2,4,4,True,False,True,0,8,True,True
9,green vegetable,vegetable,"['green pepper', 'kale', 'cabbage']",cucumber,cucumber,shared,cucumber,kale,green pepper,cabbage,1,3,3,True,False,True,0,9,True,True
